In [2]:
# C1 - Imports et chargement du dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("df_final.csv", parse_dates=["Date"])

print(f"Dimensions : {df.shape}")
print(f"Période    : {df['Date'].min().date()} à {df['Date'].max().date()}")
print(f"Colonnes   : {df.columns.tolist()}")

Dimensions : (408436, 30)
Période    : 2010-03-05 à 2012-10-26
Colonnes   : ['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Week', 'Month', 'Year', 'Lag_1', 'Lag_2', 'Lag_4', 'Rolling_Mean_4', 'Rolling_Std_4', 'Black_Friday', 'Thanksgiving', 'Xmas_Week', 'Is_Promo', 'Type_A', 'Type_B', 'Type_C']


In [3]:
# C2 - Imputation 0 + flag was_negative

avant = len(df)
negatifs = (df["Weekly_Sales"] < 0).sum()

# Création du flag avant imputation
df["was_negative"] = (df["Weekly_Sales"] < 0).astype(int)

# Imputation des négatifs par 0
df["Weekly_Sales"] = df["Weekly_Sales"].clip(lower=0)

print(f"Lignes totales             : {avant:,}")
print(f"Valeurs négatives imputées : {negatifs:,} soit {negatifs/avant*100:.2f}%")
print(f"Was_negative = 1           : {df['was_negative'].sum():,}")
print(f"Weekly_Sales min après     : {df['Weekly_Sales'].min():.2f}$")

Lignes totales             : 408,436
Valeurs négatives imputées : 1,098 soit 0.27%
Was_negative = 1           : 1,098
Weekly_Sales min après     : 0.00$


- 1 098 valeurs négatives imputées à 0 — retours clients mal encodés
- Colonne was_negative créée : 1 = valeur était négative, 0 = valeur normale
- Zéros exacts (Weekly_Sales == 0) conservés — vraies semaines sans ventes
- Aucune ligne supprimée — 408 436 lignes conservées
- Weekly_Sales min après imputation : 0.00$

In [4]:
df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,...,Rolling_Mean_4,Rolling_Std_4,Black_Friday,Thanksgiving,Xmas_Week,Is_Promo,Type_A,Type_B,Type_C,was_negative
0,1,1,2010-03-05,21827.90,0,151315,46.50,2.625,0.0,0.0,...,32990.7700,12832.106391,0,0,0,0,1,0,0,0
1,1,1,2010-03-12,21043.39,0,151315,57.79,2.667,0.0,0.0,...,32216.6200,13554.047185,0,0,0,0,1,0,0,0
2,1,1,2010-03-19,22136.64,0,151315,54.58,2.720,0.0,0.0,...,25967.5950,10467.484020,0,0,0,0,1,0,0,0
3,1,1,2010-03-26,26229.21,0,151315,51.45,2.732,0.0,0.0,...,21102.8675,1222.784968,0,0,0,0,1,0,0,0
4,1,1,2010-04-02,57258.43,0,151315,62.27,2.719,0.0,0.0,...,22809.2850,2325.929203,0,0,0,0,1,0,0,0


In [5]:
# C3 - Fonction WMAE personnalisée

def wmae(y_true, y_pred, thanksgiving, black_friday, xmas_week):
    weights = pd.Series(1.0, index=y_true.index)
    weights[(thanksgiving == 1) | (black_friday == 1) | (xmas_week == 1)] = 5
    return round((weights * (y_true - y_pred).abs()).sum() / weights.sum(), 2)

- Pondération x5 : Thanksgiving, Black_Friday, Xmas_Week
- IsHoliday retiré — capte des fêtes sans impact réel sur les ventes
- Semaines normales : poids = 1

In [6]:
# C4 - Définition des features, cible et validation

df = df.sort_values("Date").reset_index(drop=True)

features = [col for col in df.columns if col not in ["Date", "Weekly_Sales"]]
target   = "Weekly_Sales"

X            = df[features]
y            = df[target]
thanksgiving = df["Thanksgiving"]
black_friday = df["Black_Friday"]
xmas_week    = df["Xmas_Week"]

tscv = TimeSeriesSplit(n_splits=5)

print(f"Nombre de features : {len(features)}")
print(f"Lignes             : {len(X):,}")
print(f"Features           : {features}")

Nombre de features : 29
Lignes             : 408,436
Features           : ['Store', 'Dept', 'IsHoliday', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Week', 'Month', 'Year', 'Lag_1', 'Lag_2', 'Lag_4', 'Rolling_Mean_4', 'Rolling_Std_4', 'Black_Friday', 'Thanksgiving', 'Xmas_Week', 'Is_Promo', 'Type_A', 'Type_B', 'Type_C', 'was_negative']


In [7]:
# C5 - Target Encoding - fonction réutilisable

def target_encoding(X_train, X_val, y_train):
    
    X_train["Store_Dept_key"] = X_train["Store"].astype(str) + "_" + X_train["Dept"].astype(str)
    X_val["Store_Dept_key"]   = X_val["Store"].astype(str)   + "_" + X_val["Dept"].astype(str)

    store_mean      = y_train.groupby(X_train["Store"].values).mean()
    dept_mean       = y_train.groupby(X_train["Dept"].values).mean()
    store_dept_mean = y_train.groupby(X_train["Store_Dept_key"].values).mean()

    X_train["Store_enc"]      = X_train["Store"].map(store_mean)
    X_train["Dept_enc"]       = X_train["Dept"].map(dept_mean)
    X_train["Store_Dept_enc"] = X_train["Store_Dept_key"].map(store_dept_mean)

    X_val["Store_enc"]      = X_val["Store"].map(store_mean)
    X_val["Dept_enc"]       = X_val["Dept"].map(dept_mean)
    X_val["Store_Dept_enc"] = X_val["Store_Dept_key"].map(store_dept_mean)

    global_mean = y_train.mean()
    X_val["Store_enc"]      = X_val["Store_enc"].fillna(global_mean)
    X_val["Dept_enc"]       = X_val["Dept_enc"].fillna(global_mean)
    X_val["Store_Dept_enc"] = X_val["Store_Dept_enc"].fillna(global_mean)

    X_train = X_train.drop(columns=["Store_Dept_key"])
    X_val   = X_val.drop(columns=["Store_Dept_key"])

    return X_train, X_val

- Moyennes calculées sur le fold train uniquement — pas de data leakage
- Appliquées sur la validation — les moyennes viennent du passé uniquement
- Sécurité : combinaisons absentes remplacées par la moyenne globale du train

In [8]:
# C6 - Boucle CV - Random Forest V3 : imputation 0 + flag was_negative

resultats_v3 = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X), 1):

    X_train = X.iloc[train_idx].copy()
    X_val   = X.iloc[val_idx].copy()
    y_train = y.iloc[train_idx]
    y_val   = y.iloc[val_idx]

    # Variables WMAE
    t_val  = thanksgiving.iloc[val_idx]
    bf_val = black_friday.iloc[val_idx]
    xw_val = xmas_week.iloc[val_idx]

    # Target Encoding
    X_train, X_val = target_encoding(X_train, X_val, y_train)

    # Modèle
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred   = model.predict(X_val)
    y_pred_s = pd.Series(y_pred, index=y_val.index)

    # Métriques
    resultats_v3.append({
        "Variante" : "V3 — Imputation 0 + flag + Target Encoding",
        "Fold"     : f"Fold {fold}",
        "WMAE"     : wmae(y_val, y_pred_s, t_val, bf_val, xw_val),
        "MAE"      : round(mean_absolute_error(y_val, y_pred), 2),
        "RMSE"     : round(np.sqrt(mean_squared_error(y_val, y_pred)), 2),
        "R2"       : round(r2_score(y_val, y_pred), 4),
    })

    print(f"Fold {fold} — WMAE: {resultats_v3[-1]['WMAE']:,.0f}$ "
          f"MAE: {resultats_v3[-1]['MAE']:,.0f}$ "
          f"RMSE: {resultats_v3[-1]['RMSE']:,.0f}$ "
          f"R²: {resultats_v3[-1]['R2']:.4f}")

Fold 1 — WMAE: 4,286$ MAE: 2,878$ RMSE: 9,833$ R²: 0.8350
Fold 2 — WMAE: 1,527$ MAE: 1,527$ RMSE: 3,320$ R²: 0.9763
Fold 3 — WMAE: 2,566$ MAE: 1,757$ RMSE: 6,573$ R²: 0.9208
Fold 4 — WMAE: 1,893$ MAE: 1,734$ RMSE: 3,686$ R²: 0.9749
Fold 5 — WMAE: 1,363$ MAE: 1,363$ RMSE: 2,815$ R²: 0.9837


In [9]:
# C7 - Résultats et export V3

df_v3_results = pd.DataFrame(resultats_v3)

# Moyennes
print("--- Moyennes V3 ---")
print(df_v3_results[["WMAE","MAE","RMSE","R2"]].mean().round(2))

# Export
df_v3_results.to_csv("results_v3.csv", index=False)
print("\nExport results_v3.csv réussi")

--- Moyennes V3 ---
WMAE    2327.02
MAE     1851.66
RMSE    5245.46
R2         0.94
dtype: float64

Export results_v3.csv réussi
